In [ ]:
import polars as pl
from transformers import AutoTokenizer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
DATA = "../data/raw/tweets.csv"
CLEANED_DATA = "../data/processed/trump_cleaned.parquet"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
token_limit = tokenizer.model_max_length

In [ ]:
raw_df = pl.read_csv(DATA)
raw_df.head()

In [ ]:
cleaned_df = pl.read_parquet(CLEANED_DATA)
cleaned_df

In [ ]:
# Removes URLs
# Needed to compute coherence
def clean_tweet(col: pl.Expr) -> pl.Expr:
    return (
        col.str.extract_all(r"\S+")  # Split into list of words
        .list.eval(
            pl.element().filter(
                # Remove links (http...) AND words containing 'covfefe'
                ~pl.element().str.starts_with("http") & 
                # Removes covfefe as well, as the problem is fundamentally words
                # which are not in the model's dictionary
                ~pl.element().str.to_lowercase().str.contains("covfefe")
            )
        )
        .list.join(" ")  # Join back to string
    )

In [ ]:
# Clean dataset
df = raw_df.with_columns(
    # Binary
    (pl.col("isRetweet").str.to_lowercase() == "t").alias("is_retweet"),
    (pl.col("isDeleted").str.to_lowercase() == "t").alias("is_deleted"),
    (pl.col("isFlagged").str.to_lowercase() == "t").alias("is_flagged"),
    # Date
    pl.col("date").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    # Skewed numerical variables
    (pl.col("retweets") + 1).log().alias("log_retweets"),
    (pl.col("favorites") + 1).log().alias("log_favorites"),
    # Categorical variables
    pl.col("device").cast(pl.Categorical),

    # Clean text
    clean_tweet(pl.col("text"))
        .alias("clean_text"),

    # Clean + Lowercase
    clean_tweet(pl.col("text"))
        .str.to_lowercase()
        .alias("clean_text_lower"),

    # Clean + Lowercase + Remove Punctuation
    clean_tweet(pl.col("text"))
        .str.to_lowercase()
        .str.replace_all(r"[^\w\s]", "")
        .alias("clean_text_lower_punctless")
).drop(["isRetweet", "isDeleted", "isFlagged"]).with_row_index()
df.head()

In [ ]:
col = "device"
df[col].value_counts().sort(by="count", descending=True)

In [ ]:
(df["retweets"] + 1).log().describe()

In [ ]:
#df = df.with_columns(
df = cleaned_df.with_columns(
    pl.col("text")
    .map_elements(
        lambda s: len(tokenizer.encode(s, add_special_tokens=True)),
        return_dtype=pl.Int64
    ).alias("token_count")
)
df.head()

In [ ]:
exceeded_df = df.filter(pl.col("token_count") > token_limit)
exceeded_df

In [ ]:
df.write_parquet("../data/processed/trump_cleaned.parquet")

In [ ]:
embeddings_df = pl.read_parquet("../data/processed/trump_embeddings.parquet")
embeddings_df

In [ ]:
embeddings_df.filter(pl.col("text").is_null())

In [ ]:
interim_df = pl.read_parquet("../data/interim/trump_processed.parquet")
interim_df["clean_text_with_metadata"][0]